# マルチセンター研究用動画処理と画像選定パイプライン

このノートブックは、マルチセンター研究用の動画処理と画像選定パイプラインです。

## 処理フロー

1. **動画からフレーム抽出**: "E:\Multicenter_ROP_study\Multicenter_movies"内の動画を5フレーム毎にPNGに分解
2. **品質評価**: 各動画の画像に対してvalidate_images_disc.ipynbのアルゴリズムを適用
3. **ベスト画像選出**: 各動画ごとにベスト画像を選出してコピー（30枚に満たなくてもOK）
4. **lens_image保存**: 選出された画像のlens_image（Lens bbox内で中央円外を灰色塗りつぶし）も保存
5. **Excel出力**: 全動画のベスト画像を1つのExcelファイルにまとめて出力

---

## 画像選定アルゴリズム（Disc Edge Coverage + Retina Ratio版）

### 基本方針

1. **絶対足切り**: `retina_ratio < 30` の画像は選ばない
2. **Stage 1（優先）**: `disc_edge_coverage_ratio >= 0.80` AND `retina_ratio >= 30` → スコアでソート
3. **Stage 2（補完）**: `disc_edge_coverage < 0.80` だけど `retina_ratio >= 30` → retina_ratioでソートして補完
4. ※30枚に満たなくてもOK

### スコア計算式（Stage 1のみ）

```
score = 0.4 × retina_ratio_norm + 0.4 × mbss_Grad_p90_norm + 0.2 × mbss_score_norm
```

各指標はMin-Max正規化（0-1）後に重み付け。

| 指標 | 説明 | 重み |
|------|------|------|
| retina_ratio_norm | 網膜面積比（大きいほど上位） | 0.4 |
| mbss_Grad_p90_norm | 勾配強度90パーセンタイル（高いほど上位） | 0.4 |
| mbss_score_norm | ピント品質スコア（高いほど上位） | 0.2 |

### 出力ディレクトリ

- **all_images**: 全抽出画像
- **all_lens_images**: 全抽出画像のlens_image
- **selected_images**: 選出されたベスト画像
- **selected_lens_images**: 選出されたベスト画像のlens_image

---

## 使用方法

各セルを上から順に実行してください。

In [1]:
# 共通インポート
import os
import sys
from pathlib import Path
from typing import List, Optional, Dict, Any
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from ultralytics import RTDETR, YOLO
import shutil
from datetime import datetime

# YOLO入力幅（ノートブック内で固定）
YOLO_INPUT_WIDTH = 640


## 1. 動画フレーム抽出モジュール

動画から指定間隔でフレームを抽出する関数を定義します。


In [2]:
# ==================== 動画フレーム抽出モジュール ====================

def extract_frames_from_video(
    video_path: str,
    output_dir: str,
    frame_interval: int = 5,
    image_prefix: str = None
) -> List[str]:
    """
    動画から指定間隔でフレームを抽出してPNGとして保存
    
    Args:
        video_path: 動画ファイルのパス
        output_dir: 出力ディレクトリ
        frame_interval: 抽出間隔（デフォルト: 5フレーム毎）
        image_prefix: 画像ファイル名のプレフィックス（Noneの場合は動画のbasenameを使用）
    
    Returns:
        抽出された画像ファイルのパスのリスト
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # 画像ファイル名のプレフィックスを決定
    if image_prefix is None:
        image_prefix = Path(video_path).stem
    
    # 動画の読み込み
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise ValueError(f"動画を開けませんでした: {video_path}")
    
    # 総フレーム数の取得
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    extracted_images = []
    frame_count = 0
    saved_count = 0
    
    # 進捗バーを表示しながらフレーム抽出
    with tqdm(total=total_frames, desc=f"フレーム抽出: {Path(video_path).name}") as pbar:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            
            # 指定間隔ごとに保存
            if frame_count % frame_interval == 0:
                image_filename = f"{image_prefix}_{saved_count:04d}.png"
                image_path = output_dir / image_filename
                cv2.imwrite(str(image_path), frame)
                extracted_images.append(str(image_path))
                saved_count += 1
            
            frame_count += 1
            pbar.update(1)
    
    cap.release()
    
    print(f"合計 {saved_count} フレームを抽出しました")
    return extracted_images


def find_video_files(
    root_dir: str,
    extensions: tuple = ('.mov', '.mp4'),  # Windowsは大文字小文字を区別しないため小文字のみ
    recursive: bool = False  # デフォルトは直下のみ検索
) -> List[str]:
    """
    指定ディレクトリ内の動画ファイルを検索
    
    Args:
        root_dir: 検索対象のルートディレクトリ
        extensions: 対象となる拡張子のタプル（小文字のみ指定、Windowsでは大文字小文字を区別しない）
        recursive: Trueの場合サブディレクトリも再帰的に検索、Falseの場合は直下のみ
    
    Returns:
        動画ファイルのパスのリスト
    """
    root_path = Path(root_dir)
    if not root_path.exists():
        raise ValueError(f"ディレクトリが存在しません: {root_dir}")
    
    video_files = []
    
    for ext in extensions:
        if recursive:
            # サブディレクトリも再帰的に検索
            for video_path in root_path.rglob(f"*{ext}"):
                video_files.append(str(video_path))
        else:
            # 直下のみ検索
            for video_path in root_path.glob(f"*{ext}"):
                video_files.append(str(video_path))
    
    return sorted(video_files)

## 2. 画像品質評価モジュール

validate_images.ipynbのアルゴリズムを移植した品質評価関数を定義します。


In [3]:
# ==================== 画像品質特徴量（MBSS） ====================

def to_gray_float(img_bgr_or_gray: np.ndarray) -> np.ndarray:
    """BGR/Gray いずれも float32 [0,1] グレースケールへ"""
    if img_bgr_or_gray.ndim == 3:
        gray = cv2.cvtColor(img_bgr_or_gray, cv2.COLOR_BGR2GRAY)
    else:
        gray = img_bgr_or_gray
    gray = gray.astype(np.float32)
    if gray.max() > 1.0:
        gray /= 255.0
    return gray


def laplacian_multi_var(gray01: np.ndarray, sigmas=(1.0, 2.0, 4.0), weights=(0.5, 0.3, 0.2)) -> float:
    """マルチスケール Laplacian 分散（重み付き和）"""
    vals = []
    for s, w in zip(sigmas, weights):
        ksize = int(6 * s + 1)
        if ksize % 2 == 0:
            ksize += 1
        blur = cv2.GaussianBlur(gray01, (ksize, ksize), s)
        lap = cv2.Laplacian(blur, cv2.CV_32F, ksize=3)
        vals.append(w * float(lap.var()))
    return float(np.sum(vals))


def fft_features(gray01: np.ndarray, high_freq_thresh=0.3) -> tuple:
    """FFT高周波エネルギー比とスペクトル重心（NumPy FFT版）"""
    h, w = gray01.shape

    wy = np.hanning(h).astype(np.float32)
    wx = np.hanning(w).astype(np.float32)
    window = np.outer(wy, wx)
    g = gray01 * window

    F = np.fft.fftshift(np.fft.fft2(g))
    mag2 = (np.abs(F) ** 2).astype(np.float64)

    cy, cx = h // 2, w // 2
    yy, xx = np.indices((h, w))
    ry = (yy - cy) / float(max(cy, 1))
    rx = (xx - cx) / float(max(cx, 1))
    r = np.sqrt(rx ** 2 + ry ** 2)
    r_norm = np.clip(r, 0, 1)

    total = mag2.sum() + 1e-12
    high_mask = r_norm > high_freq_thresh
    hf_ratio = float(mag2[high_mask].sum() / total)
    spec_centroid = float((r_norm * mag2).sum() / total)
    return hf_ratio, spec_centroid


def grad_percentile(gray01: np.ndarray, p=90) -> float:
    """勾配強度のパーセンタイル"""
    gx = cv2.Sobel(gray01, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray01, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx ** 2 + gy ** 2)
    return float(np.percentile(mag, p))


def compute_mbss_components(img_bgr: np.ndarray, mask01: Optional[np.ndarray] = None) -> Dict[str, Any]:
    """Retina領域（mask）内のみで MBSS コンポーネントを算出

    NOTE:
    - 既存のMBSS系（Laplacian/FFT/Grad）はマスク外を0にして計算
    - 色調用に、網膜マスク内の彩度平均 `S_mean`（HSVのS）も同時に返す
    """
    gray = to_gray_float(img_bgr)

    mask_bool = None
    if mask01 is not None:
        if mask01.shape != gray.shape:
            mask01 = cv2.resize(mask01.astype(np.uint8), (gray.shape[1], gray.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask_bool = mask01 > 0
        if mask_bool.sum() < 100:
            return {"L_multi": None, "HF_ratio": None, "Spec_centroid": None, "Grad_p90": None, "S_mean": None}
        gray2 = gray.copy()
        gray2[~mask_bool] = 0.0
    else:
        gray2 = gray

    # --- 色調（HSV彩度Sの平均、網膜マスク内） ---
    s_mean = None
    if mask_bool is not None and img_bgr is not None and getattr(img_bgr, "ndim", 0) == 3:
        hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
        s = hsv[:, :, 1].astype(np.float32)
        if s.max() > 1.0:
            s /= 255.0
        roi = s[mask_bool]
        if roi.size > 0:
            s_mean = float(np.mean(roi))

    return {
        "L_multi": laplacian_multi_var(gray2),
        "HF_ratio": fft_features(gray2)[0],
        "Spec_centroid": fft_features(gray2)[1],
        "Grad_p90": grad_percentile(gray2),
        "S_mean": s_mean,
    }


def compute_mbss_score(components: dict, stats: dict, weights=None) -> Optional[float]:
    """z-score正規化後、重み付き和でスコア化"""
    if any(components.get(k) is None for k in ["L_multi", "HF_ratio", "Spec_centroid", "Grad_p90"]):
        return None

    if weights is None:
        weights = {"L_multi": 0.35, "HF_ratio": 0.25, "Spec_centroid": 0.20, "Grad_p90": 0.20}

    score = 0.0
    for k, w in weights.items():
        x = float(components[k])
        m = float(stats[k]["mean"])
        s = float(stats[k]["std"]) + 1e-8
        z = (x - m) / s
        score += w * z
    return float(score)


In [4]:
# ==================== Disc Edge Coverage（辺縁被覆率） ====================

def compute_disc_edge_coverage(disc_mask: np.ndarray, retina_mask: np.ndarray) -> tuple:
    """
    Discの辺縁がRetinaマスクに覆われているかを計算

    Args:
        disc_mask: Discマスク（0/255 or 0/1）
        retina_mask: Retinaマスク（0/255 or 0/1）

    Returns:
        tuple: (disc_edge_covered: bool, disc_edge_coverage_ratio: float)
               - disc_edge_covered: True if coverage >= 95%
               - disc_edge_coverage_ratio: 0-1の被覆率
    """
    if disc_mask is None or retina_mask is None:
        return None, None

    disc_bin = (disc_mask > 0).astype(np.uint8)
    retina_bin = (retina_mask > 0).astype(np.uint8)

    if disc_bin.sum() == 0:
        return None, None

    # Discマスクの輪郭（辺縁）を抽出
    kernel = np.ones((3, 3), np.uint8)
    disc_eroded = cv2.erode(disc_bin, kernel, iterations=1)
    disc_edge = disc_bin - disc_eroded

    total_edge_pixels = disc_edge.sum()
    if total_edge_pixels == 0:
        return None, None

    # Retinaマスクを少し膨張させて境界付近でも検出
    retina_dilated = cv2.dilate(retina_bin, kernel, iterations=2)
    covered_edge_pixels = (disc_edge & retina_dilated).sum()

    coverage_ratio = covered_edge_pixels / total_edge_pixels
    is_covered = coverage_ratio >= 0.95

    return is_covered, float(coverage_ratio)


# ==================== Disc周囲（core/ring）評価 ====================

def estimate_disc_center_radius(disc_mask01: np.ndarray):
    """discマスクから中心(cx,cy)と代表半径Rを推定"""
    m = disc_mask01.astype(np.uint8)
    if m.max() > 1:
        m = (m > 0).astype(np.uint8)

    num_labels, labels = cv2.connectedComponents(m)
    if num_labels > 1:
        areas = [(labels == i).sum() for i in range(1, num_labels)]
        main_label = int(np.argmax(areas) + 1)
        m = (labels == main_label).astype(np.uint8)

    M = cv2.moments(m)
    if M["m00"] == 0:
        return None

    cx = M["m10"] / M["m00"]
    cy = M["m01"] / M["m00"]
    area = float(m.sum())
    R = float(np.sqrt(area / np.pi))
    return cx, cy, R


def make_disc_rois(shape_hw, cx, cy, R, inner_ratio=0.6, outer_ratio=1.2):
    """Discのcore/ring領域を作成"""
    h, w = shape_hw
    yy, xx = np.indices((h, w))
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    core = dist < (inner_ratio * R)
    ring = (dist >= (inner_ratio * R)) & (dist < (outer_ratio * R))
    return core.astype(np.uint8), ring.astype(np.uint8)


def laplacian_multi_var_masked(gray01: np.ndarray, mask01: np.ndarray, sigmas=(1.0, 2.0, 4.0), weights=(0.5, 0.3, 0.2)) -> float:
    """マスク内でのマルチスケールLaplacian分散"""
    mask_bool = mask01.astype(bool)
    if mask_bool.sum() < 50:
        return 0.0

    vals = []
    for s, w in zip(sigmas, weights):
        ksize = int(6 * s + 1)
        if ksize % 2 == 0:
            ksize += 1
        blur = cv2.GaussianBlur(gray01, (ksize, ksize), s)
        lap = cv2.Laplacian(blur, cv2.CV_32F, ksize=3)
        roi = lap[mask_bool]
        if roi.size == 0:
            continue
        vals.append(w * float(roi.var()))
    return float(np.sum(vals)) if vals else 0.0


def compute_disc_sharpness_components(img_bgr: np.ndarray, disc_mask01: np.ndarray):
    """disc中心(core)と周辺(ring)の L_multi を返す"""
    gray = to_gray_float(img_bgr)
    est = estimate_disc_center_radius(disc_mask01)
    if est is None:
        return None, None

    cx, cy, R = est
    core_mask, ring_mask = make_disc_rois(gray.shape, cx, cy, R)

    if core_mask.sum() < 50 or ring_mask.sum() < 50:
        return None, None

    L_core = laplacian_multi_var_masked(gray, core_mask)
    L_ring = laplacian_multi_var_masked(gray, ring_mask)
    return L_core, L_ring

In [5]:
# ==================== メイン処理関数 ====================

def process_one_image(
    image_path: str,
    detection_model,
    segmentation_model,
    lens_output_dir: str = None
) -> Optional[dict]:
    """1枚の画像に対して推論 + 特徴量を算出
    
    Args:
        image_path: 入力画像のパス
        detection_model: RT-DETRモデル
        segmentation_model: YOLO-segモデル
        lens_output_dir: lens_image保存先ディレクトリ（Noneの場合は保存しない）
    
    Returns:
        評価結果のdict（lens_image_pathを含む）
    """
    image = cv2.imread(image_path)
    if image is None:
        return None

    # --- Stage 1: RT-DETRでLens bbox検出（cls=0想定） ---
    det_results = detection_model(image, verbose=False)
    lens_bbox_xyxy = None
    for r in det_results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
        for box in r.boxes:
            if int(box.cls) == 0:
                lens_bbox_xyxy = box.xyxy[0].cpu().numpy()
                break
        if lens_bbox_xyxy is not None:
            break

    if lens_bbox_xyxy is None:
        return {
            'image_path': image_path,
            'lens_image_path': None,
            'lens_detected': False,
            'lens_area': 0,
            'retina_area': 0,
            'retina_ratio': 0.0,
            'disc_detected': False,
            'macula_detected': False,
            'mbss_L_multi': None,
            'mbss_HF_ratio': None,
            'mbss_Spec_centroid': None,
            'mbss_Grad_p90': None,
            'S_mean': None,
            'disc_core_L_multi': None,
            'disc_ring_L_multi': None,
            'disc_center_dist_ratio': None,
            'disc_pos_ok': None,
            'disc_edge_covered': None,
            'disc_edge_coverage_ratio': None,
        }

    x1, y1, x2, y2 = [int(c) for c in lens_bbox_xyxy]
    cropped = image[y1:y2, x1:x2]
    if cropped.size == 0:
        return {
            'image_path': image_path,
            'lens_image_path': None,
            'lens_detected': True,
            'lens_area': 0,
            'retina_area': 0,
            'retina_ratio': 0.0,
            'disc_detected': False,
            'macula_detected': False,
            'mbss_L_multi': None,
            'mbss_HF_ratio': None,
            'mbss_Spec_centroid': None,
            'mbss_Grad_p90': None,
            'S_mean': None,
            'disc_core_L_multi': None,
            'disc_ring_L_multi': None,
            'disc_center_dist_ratio': None,
            'disc_pos_ok': None,
            'disc_edge_covered': None,
            'disc_edge_coverage_ratio': None,
        }

    # --- Lens内での円形マスク（レンズ外を灰色にする） ---
    orig_h, orig_w = cropped.shape[:2]
    center_x = orig_w // 2
    center_y = orig_h // 2
    diameter = (orig_w + orig_h) / 2
    radius = int(diameter / 2)

    circle_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
    cv2.circle(circle_mask, (center_x, center_y), radius, 255, -1)

    masked_cropped = cropped.copy()
    masked_cropped[circle_mask == 0] = (114, 114, 114)

    lens_area = int((circle_mask > 0).sum())

    # --- lens_image を保存 ---
    lens_image_path = None
    if lens_output_dir is not None:
        lens_output_path = Path(lens_output_dir)
        lens_output_path.mkdir(parents=True, exist_ok=True)
        # 元画像と同じファイル名で保存
        image_filename = Path(image_path).name
        lens_image_path = str(lens_output_path / image_filename)
        cv2.imwrite(lens_image_path, masked_cropped)

    # --- Stage 2: YOLO-seg ---
    aspect_ratio = orig_h / max(orig_w, 1)
    yolo_h = int(YOLO_INPUT_WIDTH * aspect_ratio)
    yolo_input = cv2.resize(masked_cropped, (YOLO_INPUT_WIDTH, yolo_h), interpolation=cv2.INTER_AREA)

    seg_results = segmentation_model(yolo_input, verbose=False, retina_masks=True)

    retina_area = 0
    disc_detected = False
    macula_detected = False

    retina_mask_crop = None
    disc_mask_crop = None

    if seg_results and seg_results[0].masks is not None:
        r0 = seg_results[0]
        masks = r0.masks.data.cpu().numpy()
        classes = r0.boxes.cls.cpu().numpy().astype(int)

        for mask_data, cls_id in zip(masks, classes):
            # mask_data: (H', W') 0..1
            mask_resized = cv2.resize(mask_data, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
            mask_bin = (mask_resized > 0.5) & (circle_mask > 0)

            if cls_id == 0:  # Fundus/Retina
                retina_area = int(mask_bin.sum())
                retina_mask_crop = (mask_bin.astype(np.uint8) * 255)
            elif cls_id == 1:  # Disc
                disc_detected = True
                disc_mask_crop = (mask_bin.astype(np.uint8) * 255)
            elif cls_id == 2:  # Macula
                macula_detected = True

    retina_ratio = (retina_area / lens_area * 100.0) if lens_area > 0 else 0.0

    # --- MBSS（Retina領域内） ---
    if retina_mask_crop is not None:
        mb = compute_mbss_components(cropped, mask01=retina_mask_crop)
    else:
        mb = {"L_multi": None, "HF_ratio": None, "Spec_centroid": None, "Grad_p90": None, "S_mean": None}

    # --- Disc周囲（core/ring） ---
    disc_core_L_multi = None
    disc_ring_L_multi = None
    disc_center_dist_ratio = None
    disc_pos_ok = None
    disc_edge_covered = None
    disc_edge_coverage_ratio = None
    if disc_mask_crop is not None:
        disc_core_L_multi, disc_ring_L_multi = compute_disc_sharpness_components(cropped, disc_mask_crop)
        est = estimate_disc_center_radius(disc_mask_crop)
        if est is not None:
            dcx, dcy, _ = est
            dist = ((dcx - center_x) ** 2 + (dcy - center_y) ** 2) ** 0.5
            disc_center_dist_ratio = float(dist / max(radius, 1))
            disc_pos_ok = (0.25 <= disc_center_dist_ratio <= 0.75)
        # Disc Edge Coverage計算
        disc_edge_covered, disc_edge_coverage_ratio = compute_disc_edge_coverage(disc_mask_crop, retina_mask_crop)

    return {
        'image_path': image_path,
        'lens_image_path': lens_image_path,
        'lens_detected': True,
        'lens_area': lens_area,
        'retina_area': retina_area,
        'retina_ratio': round(float(retina_ratio), 2),
        'disc_detected': bool(disc_detected),
        'macula_detected': bool(macula_detected),
        'mbss_L_multi': mb['L_multi'],
        'mbss_HF_ratio': mb['HF_ratio'],
        'mbss_Spec_centroid': mb['Spec_centroid'],
        'mbss_Grad_p90': mb['Grad_p90'],
        'S_mean': mb.get('S_mean'),
        'disc_core_L_multi': disc_core_L_multi,
        'disc_ring_L_multi': disc_ring_L_multi,
        'disc_center_dist_ratio': disc_center_dist_ratio,
        'disc_pos_ok': disc_pos_ok,
        'disc_edge_covered': disc_edge_covered,
        'disc_edge_coverage_ratio': disc_edge_coverage_ratio,
    }


def load_models(rtdetr_model_path: str, yolo_seg_model_path: str, device: str = "auto"):
    """
    モデルを読み込む（1回だけ呼び出す）

    Args:
        rtdetr_model_path: RT-DETRモデルのパス
        yolo_seg_model_path: YOLO-segモデルのパス
        device: デバイス指定（"auto", "cuda", "cpu"）

    Returns:
        tuple: (detection_model, segmentation_model)
    """
    print("モデルを読み込んでいます...")
    detection_model = RTDETR(rtdetr_model_path)
    segmentation_model = YOLO(yolo_seg_model_path)

    if device == "auto":
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if device == "cuda":
        detection_model.to('cuda')
        segmentation_model.to('cuda')
        print("CUDAを使用します")
    else:
        print("CPUを使用します")

    print("モデル読み込み完了")
    return detection_model, segmentation_model


def assess_image_quality(
    image_paths: List[str],
    detection_model,
    segmentation_model,
    image_id: str = None,
    lens_output_dir: str = None,
) -> pd.DataFrame:
    """
    画像リストに対して品質評価を実行
    
    Args:
        image_paths: 評価する画像ファイルのパスのリスト
        detection_model: 読み込み済みのRT-DETRモデル
        segmentation_model: 読み込み済みのYOLO-segモデル
        image_id: 画像ID（動画のbasenameなど）
        lens_output_dir: lens_image保存先ディレクトリ（Noneの場合は保存しない）
    
    Returns:
        品質指標を含むDataFrame
    """
    # 画像処理
    results = []
    for image_path in tqdm(image_paths, desc="画像を処理中"):
        try:
            r = process_one_image(
                image_path,
                detection_model,
                segmentation_model,
                lens_output_dir=lens_output_dir
            )
            if r is None:
                continue
            r['image_name'] = str(image_path.split('\\')[-1].split('/')[-1])  # ファイル名のみ
            if image_id is not None:
                r['image_id'] = image_id
            results.append(r)
        except Exception as e:
            print(f"エラー: {image_path}: {e}")
    
    if not results:
        raise RuntimeError("処理結果が0件です。image_dir/モデル/依存関係を確認してください")
    
    # DataFrame化
    df = pd.DataFrame(results)
    
    # -------------------- スコア算出（ID内でz-score） --------------------
    
    # MBSS stats（None除外）
    mb_cols = ['mbss_L_multi', 'mbss_HF_ratio', 'mbss_Spec_centroid', 'mbss_Grad_p90']
    stats = {}
    for c in mb_cols:
        vals = df[c].dropna().astype(float)
        key = c.replace('mbss_', '')
        if len(vals) > 0 and float(vals.std()) > 0:
            stats[key] = {"mean": float(vals.mean()), "std": float(vals.std())}
        elif len(vals) > 0:
            stats[key] = {"mean": float(vals.mean()), "std": 1.0}
    
    # MBSS score
    mbss_scores = []
    for _, row in df.iterrows():
        comps = {
            "L_multi": row.get('mbss_L_multi'),
            "HF_ratio": row.get('mbss_HF_ratio'),
            "Spec_centroid": row.get('mbss_Spec_centroid'),
            "Grad_p90": row.get('mbss_Grad_p90'),
        }
        if set(stats.keys()) == {"L_multi", "HF_ratio", "Spec_centroid", "Grad_p90"}:
            mbss_scores.append(compute_mbss_score(comps, stats=stats))
        else:
            mbss_scores.append(None)
    df['mbss_score'] = mbss_scores
    
    # Disc core/ring score（z-score）
    for col_l, col_s in [('disc_core_L_multi', 'disc_core_score'), ('disc_ring_L_multi', 'disc_ring_score')]:
        vals = df[col_l].dropna().astype(float)
        if len(vals) > 1 and float(vals.std()) > 0:
            m, s = float(vals.mean()), float(vals.std())
        elif len(vals) > 0:
            m, s = float(vals.mean()), 1.0
        else:
            m, s = 0.0, 1.0
        
        scores = []
        for v in df[col_l]:
            if v is None or pd.isna(v):
                scores.append(None)
            else:
                scores.append((float(v) - m) / (s + 1e-8))
        df[col_s] = scores
    
    return df

## 3. ベスト画像選出モジュール

品質評価結果からベスト30画像を選出する関数を定義します。

In [6]:
# ==================== ベスト画像選出モジュール（新アルゴリズム: Disc Edge Coverage版） ====================
# アルゴリズム:
# 1. 絶対足切り: retina_ratio < 30 の画像は選ばない
# 2. Stage 1: disc_edge_coverage_ratio >= 0.80 AND retina_ratio >= 30 の画像を優先（スコアでソート）
# 3. Stage 2: disc_edge_coverage < 0.80 だけど retina_ratio >= 30 の画像で補完（retina_ratioでソート）
# ※30枚に満たなくてもOK

# -------------------- パラメータ --------------------
# 足切り閾値
EDGE_COVERAGE_CUTOFF = 0.80
RETINA_RATIO_CUTOFF = 30  # retina_ratio < 30 の画像は絶対足切り

# スコア重み
WEIGHT_RETINA_RATIO = 0.4
WEIGHT_MBSS_GRAD_P90 = 0.4
WEIGHT_MBSS_SCORE = 0.2


def minmax_norm(series: pd.Series) -> pd.Series:
    """Min-Max正規化（0-1）"""
    min_val = series.min()
    max_val = series.max()
    if max_val - min_val < 1e-8:
        return pd.Series([0.5] * len(series), index=series.index)
    return (series - min_val) / (max_val - min_val)


def select_best_images(
    df: pd.DataFrame,
    top_k: int = 30,
    need_k: int = 5
) -> pd.DataFrame:
    """
    品質評価結果からベスト画像を選出（disc_edge_coverage版 + retina_ratio足切り）

    Args:
        df: 品質評価結果のDataFrame
        top_k: 最終出力数（デフォルト: 30）
        need_k: 最低必要数（デフォルト: 5）※互換性のため残すが使用しない

    Returns:
        ベスト画像のDataFrame（rank列を含む）

    Algorithm:
        絶対足切り: retina_ratio < 30 の画像は選ばない
        Stage 1: disc_edge_coverage >= 0.80 AND retina_ratio >= 30 → スコアでソート
        Stage 2: disc_edge_coverage < 0.80 だけど retina_ratio >= 30 → retina_ratioでソートして補完
        ※30枚に満たなくてもOK
    """
    # -------------------- 有効データ抽出 --------------------
    # lens_detected=True かつ retina_ratio>0
    valid = df[(df['lens_detected'] == True) & (df['retina_ratio'] > 0)].copy()

    if len(valid) == 0:
        raise RuntimeError("lens_detected=True かつ retina_ratio>0 のデータがありません")

    print(f"有効データ: {len(valid)}件")

    # -------------------- カラム補完 --------------------
    if 'mbss_score' not in valid.columns:
        valid['mbss_score'] = np.nan
    if 'mbss_Grad_p90' not in valid.columns:
        valid['mbss_Grad_p90'] = np.nan
    if 'disc_edge_coverage_ratio' not in valid.columns:
        valid['disc_edge_coverage_ratio'] = np.nan
    if 'disc_detected' not in valid.columns:
        valid['disc_detected'] = False

    # -------------------- 絶対足切り: retina_ratio >= 30 --------------------
    print(f"\n===== 絶対足切り: retina_ratio >= {RETINA_RATIO_CUTOFF} =====")
    valid_retina = valid[valid['retina_ratio'] >= RETINA_RATIO_CUTOFF].copy()
    print(f"retina_ratio >= {RETINA_RATIO_CUTOFF} の画像: {len(valid_retina)}件")

    if len(valid_retina) == 0:
        print("警告: retina_ratio >= 30 を満たす画像がありませんでした")
        empty_df = pd.DataFrame(columns=df.columns.tolist() + ['rank', 'selection_stage', 'score'])
        return empty_df

    # -------------------- Stage 1: disc_edge_coverage >= 0.80 の画像 --------------------
    print(f"\n===== Stage 1: disc_edge_coverage >= {EDGE_COVERAGE_CUTOFF} =====")

    stage1_candidates = valid_retina[
        (valid_retina['disc_detected'] == True) &
        (valid_retina['disc_edge_coverage_ratio'].notna()) &
        (valid_retina['disc_edge_coverage_ratio'] >= EDGE_COVERAGE_CUTOFF)
    ].copy()

    print(f"Stage 1 候補: {len(stage1_candidates)}件")

    if len(stage1_candidates) > 0:
        # Min-Max正規化（欠損値は0として扱う）
        stage1_candidates['retina_ratio_norm'] = minmax_norm(stage1_candidates['retina_ratio'].fillna(0))
        stage1_candidates['mbss_Grad_p90_norm'] = minmax_norm(stage1_candidates['mbss_Grad_p90'].fillna(0))
        stage1_candidates['mbss_score_norm'] = minmax_norm(stage1_candidates['mbss_score'].fillna(0))

        # スコア計算
        stage1_candidates['score'] = (
            WEIGHT_RETINA_RATIO * stage1_candidates['retina_ratio_norm'] +
            WEIGHT_MBSS_GRAD_P90 * stage1_candidates['mbss_Grad_p90_norm'] +
            WEIGHT_MBSS_SCORE * stage1_candidates['mbss_score_norm']
        )

        # スコア順にソート（降順）
        stage1_candidates = stage1_candidates.sort_values(by='score', ascending=False)
        stage1_selected = stage1_candidates.head(top_k).copy()
        stage1_selected['selection_stage'] = f'Stage1_edge_cov>={EDGE_COVERAGE_CUTOFF}'
    else:
        stage1_selected = pd.DataFrame()

    print(f"Stage 1 選定: {len(stage1_selected)}件")

    # -------------------- Stage 2: 補完（retina_ratioのみ） --------------------
    print(f"\n===== Stage 2: 補完（edge_cov < {EDGE_COVERAGE_CUTOFF} だけど retina >= {RETINA_RATIO_CUTOFF}） =====")

    n_remaining = top_k - len(stage1_selected)

    if n_remaining > 0:
        # Stage1で選ばれなかった画像から補完（retina_ratio >= 30 の画像）
        remaining = valid_retina[~valid_retina.index.isin(stage1_selected.index)].copy()

        if len(remaining) > 0:
            # retina_ratio順にソート（降順）
            remaining = remaining.sort_values(by='retina_ratio', ascending=False)
            stage2_selected = remaining.head(n_remaining).copy()
            stage2_selected['selection_stage'] = 'Stage2_補完'
            print(f"Stage 2 選定: {len(stage2_selected)}件")
        else:
            stage2_selected = pd.DataFrame()
            print("Stage 2 選定: 0件（残り画像なし）")
    else:
        stage2_selected = pd.DataFrame()
        print("Stage 2 選定: 0件（Stage 1で十分）")

    # -------------------- 結果結合 --------------------
    final_top = pd.concat([stage1_selected, stage2_selected], ignore_index=False)
    final_top = final_top.reset_index(drop=True)
    final_top['rank'] = range(1, len(final_top) + 1)

    print(f"\n===== 最終結果: {len(final_top)}件 =====")
    print(f"  Stage 1 (edge_cov>={EDGE_COVERAGE_CUTOFF}): {len(stage1_selected)}件")
    print(f"  Stage 2 (補完): {len(stage2_selected)}件")

    # -------------------- 表示 --------------------
    print(f"\n=== Best Top{min(10, len(final_top))} ===")
    for _, row in final_top.head(min(10, len(final_top))).iterrows():
        score_str = f", score={row['score']:.3f}" if 'score' in row and pd.notna(row.get('score')) else ""
        edge_cov = row.get('disc_edge_coverage_ratio')
        edge_cov_str = f", edge_cov={edge_cov:.3f}" if pd.notna(edge_cov) else ""
        stage_str = row.get('selection_stage', '')
        print(f"  {row['rank']:2d}. {row['image_name']} (retina={row['retina_ratio']:.1f}%{edge_cov_str}{score_str}) [{stage_str}]")

    if len(final_top) > 10:
        print(f"  ... (以下省略)")

    return final_top


def copy_best_images(
    best_df: pd.DataFrame,
    output_dir: str,
    output_lens_dir: str = None,
    source_column: str = 'image_path',
    lens_column: str = 'lens_image_path'
) -> List[str]:
    """
    ベスト画像を指定ディレクトリにコピー（lens_imageも同時にコピー）
    
    Args:
        best_df: ベスト画像のDataFrame
        output_dir: 出力ディレクトリ（selected_images用）
        output_lens_dir: lens_image出力ディレクトリ（Noneの場合はコピーしない）
        source_column: ソース画像パスの列名
        lens_column: lens_image画像パスの列名
    
    Returns:
        コピーされた画像ファイルのパスのリスト
    """
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    
    # lens_image用ディレクトリ
    if output_lens_dir is not None:
        output_lens_path = Path(output_lens_dir)
        output_lens_path.mkdir(parents=True, exist_ok=True)
    
    copied_paths = []
    copied_lens_paths = []
    
    for _, row in best_df.iterrows():
        # 元画像のコピー
        source_path = Path(row[source_column])
        if not source_path.exists():
            print(f"警告: ソース画像が見つかりません: {source_path}")
            continue
        
        dest_path = output_path / source_path.name
        shutil.copy2(source_path, dest_path)
        copied_paths.append(str(dest_path))
        
        # lens_imageのコピー
        if output_lens_dir is not None and lens_column in row and pd.notna(row[lens_column]):
            lens_source_path = Path(row[lens_column])
            if lens_source_path.exists():
                lens_dest_path = output_lens_path / lens_source_path.name
                shutil.copy2(lens_source_path, lens_dest_path)
                copied_lens_paths.append(str(lens_dest_path))
            else:
                print(f"警告: lens_imageが見つかりません: {lens_source_path}")
    
    print(f"{len(copied_paths)}枚の画像をコピーしました: {output_dir}")
    if output_lens_dir is not None:
        print(f"{len(copied_lens_paths)}枚のlens_imageをコピーしました: {output_lens_dir}")
    
    return copied_paths

In [7]:
# ==================== 設定 ====================

# パス設定
INPUT_VIDEO_DIR = r"E:\Multicenter_ROP_study\Multicenter_movies"
OUTPUT_IMAGES_DIR = r"E:\Multicenter_ROP_study\Multicenter_images\all_images"
OUTPUT_LENS_IMAGES_DIR = r"E:\Multicenter_ROP_study\Multicenter_images\all_lens_images"  # 全lens_image保存先
OUTPUT_SELECTED_DIR = r"E:\Multicenter_ROP_study\Multicenter_images\selected_images"
OUTPUT_SELECTED_LENS_DIR = r"E:\Multicenter_ROP_study\Multicenter_images\selected_lens_images"  # 選出されたlens_image保存先
OUTPUT_EXCEL_PATH = r"E:\Multicenter_ROP_study\Multicenter_images\selected_images.xlsx"

# モデルパス
MODELS_DIR = r"C:\Users\ykita\ROP_AI_project\ROP_project\models"
RTDETR_MODEL_PATH = os.path.join(MODELS_DIR, "rtdetr-l-1697_1703.pt")
YOLO_SEG_MODEL_PATH = os.path.join(MODELS_DIR, "yolo11n-seg_19movies.pt")

# 処理設定
FRAME_INTERVAL = 5  # 5フレーム毎に抽出
TOP_K = 30  # ベスト30を選出
NEED_K = 5  # 最低必要数（互換性のため残す）


def check_paths():
    """必要なパスとモデルファイルの存在確認"""
    errors = []
    
    # 入力ディレクトリ
    if not os.path.exists(INPUT_VIDEO_DIR):
        errors.append(f"入力動画ディレクトリが存在しません: {INPUT_VIDEO_DIR}")
    
    # モデルファイル
    if not os.path.exists(RTDETR_MODEL_PATH):
        errors.append(f"RT-DETRモデルが見つかりません: {RTDETR_MODEL_PATH}")
    
    if not os.path.exists(YOLO_SEG_MODEL_PATH):
        errors.append(f"YOLO-segモデルが見つかりません: {YOLO_SEG_MODEL_PATH}")
    
    # 出力ディレクトリは自動作成するので、親ディレクトリのみ確認
    output_images_parent = os.path.dirname(OUTPUT_IMAGES_DIR)
    if not os.path.exists(output_images_parent):
        try:
            os.makedirs(output_images_parent, exist_ok=True)
        except Exception as e:
            errors.append(f"出力ディレクトリを作成できません: {output_images_parent}: {e}")
    
    if errors:
        print("エラー: 以下の問題が見つかりました:")
        for error in errors:
            print(f"  - {error}")
        return False
    
    return True


def output_to_excel(all_best_results: List[pd.DataFrame], output_path: str):
    """
    全動画のベスト30結果を1つのExcelファイルにまとめて出力（新アルゴリズム対応）
    
    Args:
        all_best_results: 各動画のベスト30結果のDataFrameのリスト
        output_path: 出力Excelファイルのパス
    """
    if not all_best_results:
        print("警告: 出力するデータがありません")
        return
    
    # 全結果を結合
    all_df = pd.concat(all_best_results, ignore_index=True)
    
    # Excel出力用のカラムを選択・整理（新アルゴリズム対応）
    output_columns = [
        'image_id', 'rank', 'image_name', 'selection_stage',
        'retina_ratio', 'retina_area',
        'disc_detected', 'disc_edge_coverage_ratio', 'disc_edge_covered',
        'mbss_Grad_p90', 'mbss_score', 'S_mean',
        'score'
    ]
    
    # 存在するカラムのみを選択
    available_columns = [col for col in output_columns if col in all_df.columns]
    output_df = all_df[available_columns].copy()
    
    # 出力ディレクトリを作成
    output_dir = os.path.dirname(output_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    try:
        output_df.to_excel(output_path, index=False)
        print(f"\n保存しました: {output_path}")
        print(f"総行数: {len(output_df)}")
        print(f"動画数: {output_df['image_id'].nunique()}")
        
    except PermissionError as e:
        # ファイルが使用中の場合はタイムスタンプ付きで保存
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        root, ext = os.path.splitext(output_path)
        alt_path = f"{root}_{ts}{ext}"
        output_df.to_excel(alt_path, index=False)
        print(f"\n[WARN] 出力先ファイルが使用中のため上書きできませんでした: {output_path}")
        print(f"       代替ファイルに保存しました: {alt_path}")
        print(f"       総行数: {len(output_df)}")
        print(f"       動画数: {output_df['image_id'].nunique()}")
    except Exception as e:
        print(f"\nExcel出力に失敗しました: {e}")
        # 代替: CSV
        alt_csv = output_path.replace('.xlsx', '.csv')
        output_df.to_csv(alt_csv, index=False, encoding='utf-8-sig')
        print(f"代替でCSV保存しました: {alt_csv}")

## 5. メイン処理

全処理を統合して実行します。


In [8]:
# ==================== メイン処理 ====================

print("=" * 60)
print("マルチセンター研究用動画処理と画像選定パイプライン")
print("=" * 60)

# パス確認
print("\n[1/5] パスとモデルファイルの確認...")
if not check_paths():
    print("エラー: パス確認に失敗しました。処理を中断します。")
else:
    print("OK パス確認完了")
    
    # 動画ファイル検索（直下のみ、サブディレクトリは除外）
    print(f"\n[2/5] 動画ファイルを検索中: {INPUT_VIDEO_DIR}")
    video_files = find_video_files(
        INPUT_VIDEO_DIR,
        extensions=('.mov', '.mp4'),
        recursive=False  # 直下のみ検索
    )
    
    if not video_files:
        print("エラー: 処理対象の動画ファイルが見つかりませんでした")
    else:
        print(f"OK {len(video_files)}個の動画ファイルが見つかりました")
        
        # モデルを1回だけ読み込み
        print("\n[2.5/5] モデルを読み込み中...")
        detection_model, segmentation_model = load_models(
            RTDETR_MODEL_PATH, YOLO_SEG_MODEL_PATH
        )
        print("OK モデル読み込み完了")
        
        # 全動画のベスト30結果を保存するリスト
        all_best_results = []
        
        # 各動画を処理
        print(f"\n[3/5] 各動画を処理中...")
        for idx, video_path in enumerate(tqdm(video_files, desc="動画処理中", unit="動画"), 1):
            video_basename = Path(video_path).stem
            tqdm.write(f"\n--- [{idx}/{len(video_files)}] {video_basename} ---")
            
            try:
                # 2-1. フレーム抽出
                tqdm.write("  [1/4] フレーム抽出中...")
                extracted_images = extract_frames_from_video(
                    video_path=video_path,
                    output_dir=OUTPUT_IMAGES_DIR,
                    frame_interval=FRAME_INTERVAL,
                    image_prefix=video_basename
                )
                
                if not extracted_images:
                    tqdm.write(f"  警告: フレームが抽出できませんでした（スキップ）")
                    continue
                
                tqdm.write(f"  OK {len(extracted_images)}フレーム抽出完了")
                
                # 2-2. 品質評価（事前に読み込んだモデルを使用、lens_imageも保存）
                tqdm.write("  [2/4] 品質評価中（lens_imageも保存）...")
                results_df = assess_image_quality(
                    image_paths=extracted_images,
                    detection_model=detection_model,
                    segmentation_model=segmentation_model,
                    image_id=video_basename,
                    lens_output_dir=OUTPUT_LENS_IMAGES_DIR  # lens_image保存先を指定
                )
                tqdm.write(f"  OK 品質評価完了: {len(results_df)}枚の画像を評価")
                
                # 2-3. ベスト画像選出（30枚に満たなくてもOK）
                tqdm.write("  [3/4] ベスト画像選出中...")
                best_df = select_best_images(
                    df=results_df,
                    top_k=TOP_K,
                    need_k=NEED_K
                )
                
                if len(best_df) == 0:
                    tqdm.write(f"  警告: 足切り条件を満たす画像がありませんでした（スキップ）")
                    continue
                    
                tqdm.write(f"  OK ベスト{len(best_df)}枚を選出")
                
                # 2-4. selected_imagesにコピー（lens_imageも同時にコピー）
                tqdm.write("  [4/4] ベスト画像をコピー中（lens_imageも含む）...")
                copy_best_images(
                    best_df=best_df,
                    output_dir=OUTPUT_SELECTED_DIR,
                    output_lens_dir=OUTPUT_SELECTED_LENS_DIR,  # lens_imageコピー先を指定
                    source_column='image_path',
                    lens_column='lens_image_path'
                )
                tqdm.write(f"  OK コピー完了")
                
                # 2-5. 結果をリストに追加（後でExcel出力用）
                all_best_results.append(best_df)
                
                tqdm.write(f"  OK {video_basename} の処理完了")
                
            except Exception as e:
                tqdm.write(f"  エラー: {video_basename} の処理中にエラーが発生しました: {e}")
                import traceback
                traceback.print_exc()
                tqdm.write(f"  -> この動画をスキップして続行します")
                continue
        
        # Excelにまとめて出力
        print(f"\n[4/5] Excelファイルに出力中...")
        if all_best_results:
            output_to_excel(all_best_results, OUTPUT_EXCEL_PATH)
        else:
            print("警告: 出力する結果がありませんでした")
        
        # 完了
        print(f"\n[5/5] 処理完了!")
        print(f"=" * 60)
        print(f"処理した動画数: {len(all_best_results)}")
        print(f"抽出画像保存先: {OUTPUT_IMAGES_DIR}")
        print(f"全lens_image保存先: {OUTPUT_LENS_IMAGES_DIR}")
        print(f"ベスト画像保存先: {OUTPUT_SELECTED_DIR}")
        print(f"ベストlens_image保存先: {OUTPUT_SELECTED_LENS_DIR}")
        print(f"Excel出力先: {OUTPUT_EXCEL_PATH}")
        print("=" * 60)

マルチセンター研究用動画処理と画像選定パイプライン

[1/5] パスとモデルファイルの確認...
OK パス確認完了

[2/5] 動画ファイルを検索中: E:\Multicenter_ROP_study\Multicenter_movies
OK 296個の動画ファイルが見つかりました

[2.5/5] モデルを読み込み中...
モデルを読み込んでいます...
CUDAを使用します
モデル読み込み完了
OK モデル読み込み完了

[3/5] 各動画を処理中...


動画処理中:   0%|          | 0/296 [00:00<?, ?動画/s]       


--- [1/296] 0001_AMU ---
  [1/4] フレーム抽出中...


動画処理中:   0%|          | 0/296 [00:38<?, ?動画/s]       

合計 237 フレームを抽出しました
  OK 237フレーム抽出完了
  [2/4] 品質評価中（lens_imageも保存）...


...

動画処理中: 100%|█████████▉| 295/296 [12:13:46<02:30, 150.17s/動画]       

  OK 品質評価完了: 147枚の画像を評価
  [3/4] ベスト画像選出中...
有効データ: 82件

===== 絶対足切り: retina_ratio >= 30 =====
retina_ratio >= 30 の画像: 54件

===== Stage 1: disc_edge_coverage >= 0.8 =====
Stage 1 候補: 33件
Stage 1 選定: 30件

===== Stage 2: 補完（edge_cov < 0.8 だけど retina >= 30） =====
Stage 2 選定: 0件（Stage 1で十分）

===== 最終結果: 30件 =====
  Stage 1 (edge_cov>=0.8): 30件
  Stage 2 (補完): 0件

=== Best Top10 ===
   1. 0306_OWCH_0060.png (retina=79.4%, edge_cov=0.996, score=0.923) [Stage1_edge_cov>=0.8]
   2. 0306_OWCH_0059.png (retina=79.8%, edge_cov=1.000, score=0.892) [Stage1_edge_cov>=0.8]
   3. 0306_OWCH_0071.png (retina=80.8%, edge_cov=0.989, score=0.867) [Stage1_edge_cov>=0.8]
   4. 0306_OWCH_0056.png (retina=81.1%, edge_cov=1.000, score=0.805) [Stage1_edge_cov>=0.8]
   5. 0306_OWCH_0132.png (retina=81.2%, edge_cov=0.975, score=0.804) [Stage1_edge_cov>=0.8]
   6. 0306_OWCH_0125.png (retina=79.2%, edge_cov=1.000, score=0.800) [Stage1_edge_cov>=0.8]
   7. 0306_OWCH_0053.png (retina=81.8%, edge_cov=1.000, score=0.772)

動画処理中: 100%|██████████| 296/296 [12:13:46<00:00, 148.74s/動画]       

30枚の画像をコピーしました: E:\Multicenter_ROP_study\Multicenter_images\selected_images
30枚のlens_imageをコピーしました: E:\Multicenter_ROP_study\Multicenter_images\selected_lens_images
  OK コピー完了
  OK 0306_OWCH の処理完了

[4/5] Excelファイルに出力中...



保存しました: E:\Multicenter_ROP_study\Multicenter_images\selected_images.xlsx
総行数: 8701
動画数: 294

[5/5] 処理完了!
処理した動画数: 294
抽出画像保存先: E:\Multicenter_ROP_study\Multicenter_images\all_images
全lens_image保存先: E:\Multicenter_ROP_study\Multicenter_images\all_lens_images
ベスト画像保存先: E:\Multicenter_ROP_study\Multicenter_images\selected_images
ベストlens_image保存先: E:\Multicenter_ROP_study\Multicenter_images\selected_lens_images
Excel出力先: E:\Multicenter_ROP_study\Multicenter_images\selected_images.xlsx
